# Step1: QC & Preprocessing — Audit-Grade Template

**Cross-project template** for single-cell RNA-seq quality control and preprocessing with scLucid.

## Design principles

- **Step-by-step transparency**: every QC decision is visible and auditable.
- **Parameter block at top**: switch projects by editing one configuration cell.
- **Sample-aware**: multi-sample data handled correctly throughout.
- **Decision-support tools**: scLucid intelligent recommendations, stability evaluations,
  and integration diagnostics guide parameter choices.
- **Reproducibility**: contract validation at the end ensures AnnData integrity.
- **Two-mode operation**: quick-skip optional diagnostics via boolean flags.

## What this notebook covers

1. **Data Loading** — 10X or h5ad import with metadata
2. **Quality Control** — metrics → intelligent recommendations → doublet detection → threshold suggestion → adaptive marking → filtering → audit → report
3. **Preprocessing** — biotype annotation → normalization → HVG selection → regression → scaling → PCA → diagnostic embedding → integration → evaluation → final graph
4. **Contract Validation** — stage contract audit + layer/obsm integrity checks

## Required inputs

- Raw count matrix (10X directory or .h5ad file)
- A sample-level metadata key (e.g. `sampleID`)
- Optional: treatment/group metadata, gene biotype reference

## Outputs

- `Step1-sce_cleaned.h5ad` — QC-filtered AnnData
- `Step2-sce_preprocessed.h5ad` — fully preprocessed AnnData
- QC artifacts in `results/qc_*/`
- Preprocessing artifacts in `results/pp_*/`


## Environment Setup

In [ ]:
import gc
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scLucid as scl

warnings.filterwarnings("ignore")

# Logging and figure defaults
scl.setup_logging("INFO")
scl.set_figure_params(
    dpi=150,
    dpi_save=300,
    figsize=(6, 5),
    style="seaborn-v0_8",
    style_dict={"axes.grid": True},
    color_theme="default",
)

2026-06-15 15:00:13,901 - scLucid.settings - [INFO] - scLucid logging configured to level INFO.
2026-06-15 15:00:13,901 - scLucid.settings - [INFO] - Applying global plotting settings...
2026-06-15 15:00:13,902 - scLucid.settings - [INFO] - Applied matplotlib style: seaborn-v0_8
2026-06-15 15:00:13,902 - scLucid.settings - [INFO] - Applied custom style dict: {'axes.grid': True}
2026-06-15 15:00:13,904 - scLucid.settings - [INFO] - IPython inline backend set to retina.
2026-06-15 15:00:13,905 - scLucid.settings - [INFO] - Global plotting settings applied with 'default' color theme and default font.


## Parameter Configuration

**Edit this cell when switching projects.**
All paths, sample names, metadata, and analysis options are centralized here.


In [ ]:
# ==============================================================================
# PROJECT PATHS
# ==============================================================================
# Set BASE_DIR to your project root. All other paths are relative to it.

BASE_DIR = Path('/Users/luye/Library/Mobile Documents/com~apple~CloudDocs/Projects/Ongoing/202603AK112')
RAW_DIR = BASE_DIR / "RAW_DATA"           # 10X output directories (one per sample)
DATA_DIR = BASE_DIR / "1-DATA"     # Where intermediate .h5ad files are saved
RESULTS_DIR = BASE_DIR / "2-OUTPUT-AK112"         # All QC/preprocessing outputs go here

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# SPECIES & TISSUE
# ==============================================================================
SPECIES = "mouse"               # "human" or "mouse"
TISSUE = None                   # Tissue context hint, e.g. "tumor", "blood", "brain"
TISSUE_TYPE = "tumor_tissue"    # One of: "tumor_tissue", "pbmc_or_blood", "cell_line", "organoid", "unknown"

# ==============================================================================
# SAMPLES & METADATA
# ==============================================================================
# List all sample names (must match 10X subdirectory names).
ALL_SAMPLES = ['NC_1', 'NC_2', 'AK112_1', 'AK112_2']
#ALL_SAMPLES = ['A_Q20', 'A_Q42', 'P_34','P_Q19','C_Q17','C_Q24','AC_Q8','AC_Q21','AC_Q50']

# The column name that will identify samples throughout the workflow.
SAMPLE_KEY = "sampleID"

# Treatment / biological group metadata (optional).
# If provided, this is added to adata.obs during data loading.
# If you do NOT have group metadata, set GROUP_DICT = {} and GROUP_KEY = None.
GROUP_DICT = {
    "NC_1": "PBS",
    "NC_2": "PBS",
    "AK112_1": "PDL1xVEGF",   # 康方AK112：抗VEGF/PD1双抗
    "AK112_2": "PDL1xVEGF",
    #"RC148_1": "RC148",   # 荣昌RC148：抗VEGF/PD1双抗
    #"RC148_2": "RC148",
    "A_Q20": "PDL1xVEGF",
    "A_Q42": "PDL1xVEGF",
    "P_34": "PBS",
    "P_Q19":"PBS",
    'C_Q17':'CD47',
    'C_Q24':'CD47',
    'AC_Q8':'PDL1xVEGF+CD47',
    'AC_Q21':'PDL1xVEGF+CD47',
    'AC_Q50':'PDL1xVEGF+CD47'
}
GROUP_KEY = "group"  # Column name for the group label; set to None to skip

MODEL_DICT = {
    "NC_1": "CT26",
    "NC_2": "CT26",
    "AK112_1": "CT26",  
    "AK112_2": "CT26",
    "A_Q20": "SCC7",
    "A_Q42": "SCC7",
    "P_34": "SCC7",
    "P_Q19":"SCC7", 
    'C_Q17':'SCC7',
    'C_Q24':'SCC7',
    'AC_Q8':'SCC7',
    'AC_Q21':'SCC7',
    'AC_Q50':'SCC7'
}
MODEL_KEY = "model"

# Additional biology columns that should NOT be regressed out or used as integration keys.
# These represent biological signal to preserve, not technical noise.
BIOLOGY_COLUMNS = [GROUP_KEY, MODEL_KEY]  # add other biology columns here

# ==============================================================================
# PROJECT-SPECIFIC COLORS (for plots)
# ==============================================================================
# Define sample/group color mappings for consistent plot aesthetics.
# Delete or leave empty if you don't need custom colors.
SAMPLE_COLORS = {
   'NC_1': '#1E688D',
    'NC_2': "#4DB3E6",
    'AK112_1': '#55E08A',
    'AK112_2': "#267C47",
    'A_Q20': "#108A39",
    'A_Q42': '#55E0A7',
    'P_34': "#396794",
    'P_Q19': "#3A8AE6",
    'C_Q17':'#E0D855',
    'C_Q24':'#E0AB55',
    'AC_Q8':'#B155E0',
    'AC_Q21':'#E05593',
    'AC_Q50':"#73113D"
}
GROUP_COLORS = {
    "PBS": "#BFCFE3",
    "PDL1xVEGF": "#8B4C9D",
    "CD47": "#AAC72C",
    "PDL1xVEGF+CD47": "#40CCB2",
}
MODEL_COLORS = {
    "CT26": "#6B7280",
    "SCC7": "#A16207",
}

## Helper Functions

These functions handle common operations: metadata merging, sample key resolution,
integration auto-detection, and filtering audit.


In [ ]:
# Use scLucid package helpers instead of notebook-local duplicate implementations.
print_sample_crosstab = scl.utils.print_sample_crosstab
audit_doublets = scl.qc.audit_doublets
is_raw_like_counts_matrix = scl.utils.is_raw_count_matrix

---
## **Step 1: Data Loading**

Load 10X-format data using `scLucid.utils.load_10x_data()`.
This function handles multi-sample loading, metadata attachment, and saves
a combined raw `.h5ad` for reproducibility.

If your data is already in `.h5ad` format, skip to the next section and use
`sc.read_h5ad()`.


In [ ]:
# Build metadata dicts from configuration using scLucid helper
metadata_dicts = scl.utils.build_metadata_dicts(
    samples=ALL_SAMPLES,
    group_dict=GROUP_DICT,
    batch_dict=MODEL_DICT,
    group_key=GROUP_KEY,
    batch_key=MODEL_KEY,
)

# Load all samples
adata = scl.utils.load_10x_data(
    samples=ALL_SAMPLES,
    base_dir=str(RAW_DIR),
    metadata_dicts=metadata_dicts,
    output_file=str(DATA_DIR / "Step0-combined_raw_data-CT26.h5ad"),
)

print(f"Loaded {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print_sample_crosstab(adata, sample_key=SAMPLE_KEY, group_key=GROUP_KEY)
adata

---
## **Step 2: Quality Control (QC)**

### QC strategy overview

scLucid provides a layered QC approach:

1. **Calculate metrics** — n_genes, n_counts, pct_mito, pct_ribo, cell cycle scores
2. **Intelligent recommendations** — data-driven threshold suggestions with confidence intervals
3. **Doublet detection** — algorithmic + heuristic marker-based, merged via ensemble
4. **Threshold suggestion** — MAD-based or GMM-based global thresholds
5. **Adaptive marking** — per-sample adaptive thresholds, catching sample-specific outliers
6. **Unified marking** — combine all outlier flags + doublet calls into a single decision
7. **Filter + audit** — apply filter and produce retention statistics
8. **Report** — generate text and optional HTML QC report

Each step can be inspected before proceeding to the next.


In [ ]:
# ==============================================================================
# QC OPTIONS
# ==============================================================================
DOUBLET_METHOD = "doubletdetection"  # "scrublet", "doubletdetection", or "solo"
THRESHOLD_METHOD = "mad"            # "mad" (median absolute deviation), "gmm", "percentile"
MAD_MULTIPLIERS = [3, 4, 5]
MAX_CELLS_FOR_PLOTTING = 100000    # Downsample for plot performance

# Adaptive marking
ADAPTIVE_BATCH_KEY = "sampleID"      # Key for per-sample adaptive thresholding
ADAPTIVE_METRICS = ["n_genes_by_counts", "total_counts", "pct_counts_mt"]
ADAPTIVE_METHOD = "hierarchical"     # "hierarchical" or "independent"

# Manual threshold overrides (set to None to use data-driven recommendations).
MANUAL_MIN_GENES = 500              # Absolute floor for n_genes
MANUAL_MAX_GENES = None             # Override max_genes recommendation
MANUAL_MIN_COUNTS = None
MANUAL_MAX_COUNTS = None            # Override max_counts recommendation
MANUAL_PC_MT = 10                 # Override pct_mito recommendation (e.g. 10.0)
MANUAL_NMADS = 5.0                  # nMADs for outlier detection

# ==============================================================================
# REPORTING
# ==============================================================================
GENERATE_HTML_REPORT = False        # Generate interactive HTML QC report (requires plotly)

print("Configuration loaded.")
print(f"  Project: {BASE_DIR}")
print(f"  Species: {SPECIES}")
print(f"  Samples: {len(ALL_SAMPLES)} ({SAMPLE_KEY})")
print(f"  Tissue type: {TISSUE_TYPE}")

In [ ]:
gc.collect()

# Load the combined raw data
adata = sc.read_h5ad(str(DATA_DIR / "Step0-combined_raw_data-CT26.h5ad"))
print(f"Dataset loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes")

print_sample_crosstab(adata, sample_key=SAMPLE_KEY, group_key=GROUP_KEY)
print(f"\n{'='*60}")
print("Quality Control")
print(f"{'='*60}")

### 2.1 Calculate QC Metrics & Cell Cycle

Computes per-cell quality metrics:
- `n_genes_by_counts`, `total_counts` — library complexity and depth
- `pct_counts_mt`, `pct_counts_ribo`, `pct_counts_hb` — contamination indicators
- `S_score`, `G2M_score`, `phase` — cell cycle scoring (canonical markers)

Violin plots and scatter plots are generated for visual inspection.


In [ ]:
reporting_config = scl.qc.MetricsReportingConfig(
    plot_scatter=False,
    save_dir=str(RESULTS_DIR / "qc_metrics"),
)

adata = scl.qc.calculate_qc_metric(
    adata,
    sample_key=SAMPLE_KEY,
    calculate_cell_cycle=True,
    cell_cycle_species=SPECIES,
    max_cells_for_plotting=MAX_CELLS_FOR_PLOTTING,
    reporting_config=reporting_config,
)

print("QC metrics calculated.")
print(f"  Metrics in adata.obs: {[c for c in adata.obs.columns if c.startswith(('n_', 'total_', 'pct_', 'S_score', 'G2M_score', 'phase'))]}")


---
### Ambient RNA / Empty Droplet Diagnostics

These Python-only scLucid diagnostics do not perform correction. They record
ambient/background risk so downstream QC decisions remain explicit and auditable.


In [ ]:

print("---- Ambient RNA / Empty Droplet Diagnostics ----")

ambient_summary = scl.qc.diagnose_ambient_rna(
    adata,
    layer="counts" if "counts" in adata.layers else None,
)
empty_droplet_summary = scl.qc.diagnose_empty_droplets(
    adata,
    layer="counts" if "counts" in adata.layers else None,
)

print("Ambient RNA diagnostic:")
print(f"  available: {ambient_summary.get('available')}")
print(f"  risk: {ambient_summary.get('risk_level')} ({ambient_summary.get('risk_score')})")
print(f"  note: {ambient_summary.get('method_note')}")

print("Empty droplet diagnostic:")
print(f"  available: {empty_droplet_summary.get('available')}")
print(f"  risk: {empty_droplet_summary.get('risk_level')} ({empty_droplet_summary.get('risk_score')})")
print(f"  note: {empty_droplet_summary.get('method_note')}")


### 2.2 Intelligent QC Recommendations

scLucid's **IntelligentQCRecommender** analyzes the distribution of QC metrics and
proposes data-driven thresholds using multiple methods (GMM, bootstrapping, distribution fitting).

This is a core innovation of scLucid — rather than using fixed cutoffs (e.g. "n_genes > 200"),
the recommender inspects your data and provides thresholds with **confidence intervals**.

The output includes:
- A `QCRecommendation` object with threshold suggestions and confidence intervals
- Diagnostic plots showing the underlying distribution fits
- Per-sample recommendations when sample_key is provided


In [ ]:
from scLucid.qc import recommend_intelligent_qc

print("Running Intelligent QC Recommender...")
intelligent_recommendations = recommend_intelligent_qc(
    adata,
    tissue_type=TISSUE_TYPE,
    strategy="auto",
    plot=True,
    save_dir=str(RESULTS_DIR / "qc_intelligent"),
)

# Access recommendation attributes directly (dataclass, not methods)
rec = intelligent_recommendations
print(f"\nOverall confidence: {rec.overall_confidence:.2f}")
print(f"Data quality score: {rec.data_quality_score:.1f}/100")
print(f"Strategy: {rec.overall_strategy.value}")
print(f"\nRecommended thresholds (with 95% CI):")
print(f"  min_genes:      {rec.min_genes.threshold} "
      f"[{rec.min_genes.ci_lower}, {rec.min_genes.ci_upper}]")
print(f"  max_mt_percent: {rec.max_mt_percent.threshold} "
      f"[{rec.max_mt_percent.ci_lower}, {rec.max_mt_percent.ci_upper}]")
print(f"  n_counts:       {rec.n_counts.threshold} "
      f"[{rec.n_counts.ci_lower}, {rec.n_counts.ci_upper}]")
print(f"  doublet_score:  {rec.doublet_threshold.threshold} "
      f"[{rec.doublet_threshold.ci_lower}, {rec.doublet_threshold.ci_upper}]")

if rec.concerns:
    print(f"\nConcerns:")
    for c in rec.concerns:
        print(f"  - {c}")

if rec.tumor_specific_considerations:
    print(f"\nTumor-specific considerations:")
    for c in rec.tumor_specific_considerations:
        print(f"  - {c}")

# Build thresholds dict for downstream use (keys match suggest_qc_thresholds naming)
rec_thresholds = {
      "min_genes":   rec.min_genes.threshold,         # lower bound
      "max_genes":   None,                            # not provided
      "min_counts":  rec.n_counts.threshold,          # lower bound — NOT a max!
      "max_counts":  None,                            # not provided; MAD fills this
      "pc_mt":       rec.max_mt_percent.threshold,    # upper bound
  }
print(f"\nExtracted thresholds for downstream use:")
for k, v in rec_thresholds.items():
    print(f"  {k}: {v}")

### 2.3 Doublet Detection

scLucid supports multiple doublet detection methods:
- **scrublet** — fast, heuristic-based (good default for most data)
- **doubletdetection** — more sensitive, uses co-expression patterns
- **solo** — deep learning based (requires scvi-tools)

The recommended approach is to:
1. Auto-calculate expected doublet rates from the number of recovered cells per sample
2. Run algorithmic detection with the selected method
3. Enable heuristic marker-based detection as an independent check
4. Merge results via weighted averaging (`merge_strategy="weighted_average"`)


In [ ]:
print("---- Doublet Detection ----")

# Auto-calculate expected doublet rates per sample
doublet_rates = scl.qc.generate_doublet_rates(adata, sample_key=SAMPLE_KEY)
print(f"Expected doublet rates: {doublet_rates}")

doublet_cfg = scl.qc.DoubletConfig(
    run_algorithm=True,
    method=DOUBLET_METHOD,
    expected_doublet_rate=doublet_rates,
    use_heuristics=True,
    marker_species=SPECIES,
    merge_strategy="weighted_average",
    algorithm_weight=0.75,
    plot_summary=True,
    plot_bar=True,
    plot_scatter=True,
    plot_upset=False,
    show_plots=True,
    save_dir=str(RESULTS_DIR / "qc_doublets"),
)

adata = scl.qc.predict_doublets(
    adata,
    config=doublet_cfg,
    sample_key=SAMPLE_KEY,
)

# Quick summary
for col in ["scrublet_predicted", "doubletdetection_predicted", "heuristic_predicted", "predicted_doublet"]:
    if col in adata.obs.columns:
        n = adata.obs[col].sum()
        print(f"  {col}: {n} cells ({n/adata.n_obs*100:.2f}%)")

### 2.4 Suggest QC Thresholds

Computes global reference thresholds using the specified method (MAD, GMM, or percentile).
These thresholds serve as a reference; per-sample adaptive marking is applied in the next step.

The MAD method identifies outliers based on median absolute deviation, which is robust
to non-normal distributions common in scRNA-seq data.


In [ ]:
print("\n---- Suggesting QC Thresholds ----")

threshold_table, suggested_thresholds = scl.qc.suggest_qc_thresholds(
    adata,
    method=THRESHOLD_METHOD,
    mad_multipliers=MAD_MULTIPLIERS,
    plot_distributions=True,
    save_dir=str(RESULTS_DIR / "qc_metrics"),
)

print("Suggested global QC thresholds:")
display(threshold_table)
print(suggested_thresholds.to_dict())

In [ ]:
# Merge data-driven and manual QC threshold sources using scLucid's canonical resolver.
# Policy:
# - IntelligentQCRecommender has priority over MAD-based suggestions.
# - Manual min thresholds act as floors.
# - Manual max/MT thresholds act as ceilings when provided.
manual_thresholds = {
    "min_genes": MANUAL_MIN_GENES,
    "min_counts": MANUAL_MIN_COUNTS,
    "max_genes": MANUAL_MAX_GENES,
    "max_counts": MANUAL_MAX_COUNTS,
    "pc_mt": MANUAL_PC_MT,
}
mad_thresholds = suggested_thresholds.to_dict() if hasattr(suggested_thresholds, "to_dict") else dict(suggested_thresholds or {})
resolved_thresholds = scl.qc.resolve_qc_thresholds(
    intelligent=rec_thresholds,
    mad=mad_thresholds,
    manual=manual_thresholds,
    policy="intelligent_then_mad",
)

# Project defaults for required marking thresholds when neither intelligent nor MAD suggestions exist.
final_min_genes = resolved_thresholds.min_genes if resolved_thresholds.min_genes is not None else 300
final_min_counts = resolved_thresholds.min_counts
final_max_genes = resolved_thresholds.max_genes
final_max_counts = resolved_thresholds.max_counts
final_pc_mt = resolved_thresholds.pc_mt if resolved_thresholds.pc_mt is not None else 10.0

print(f"\nFinal thresholds for cell marking:")
print(f"  min_genes  = {final_min_genes}   (intelligent={rec_thresholds.get('min_genes')}, MAD={mad_thresholds.get('min_genes')}, manual_floor={MANUAL_MIN_GENES})")
print(f"  min_counts = {final_min_counts}  (intelligent={rec_thresholds.get('min_counts')}, MAD={mad_thresholds.get('min_counts')}, manual_floor={MANUAL_MIN_COUNTS})")
print(f"  max_genes  = {final_max_genes}   (intelligent={rec_thresholds.get('max_genes')}, MAD={mad_thresholds.get('max_genes')}, manual_ceiling={MANUAL_MAX_GENES})")
print(f"  max_counts = {final_max_counts}  (intelligent={rec_thresholds.get('max_counts')}, MAD={mad_thresholds.get('max_counts')}, manual_ceiling={MANUAL_MAX_COUNTS})")
print(f"  pc_mt      = {final_pc_mt}       (intelligent={rec_thresholds.get('pc_mt')}, MAD={mad_thresholds.get('pc_mt')}, manual_ceiling={MANUAL_PC_MT})")


### 2.5 Mark Low-Quality Cells (Sample-Aware + Adaptive)

Two-stage marking:
1. **Adaptive** — per-sample outlier detection using hierarchical or independent method.
   This catches cells that are outliers *within their own sample*, which is essential
   for multi-sample data where overall distributions may differ.
2. **Unified** — combines adaptive flags, global MAD-based flags, and doublet calls
   into a comprehensive set of boolean columns in `adata.obs`.

The columns_to_plot list should be updated based on which doublet method was run.


In [ ]:
print("\n---- Adaptive Cell Marking (sample-aware) ----")

adata = scl.qc.mark_low_quality_cells_adaptive(
    adata,
    batch_key=ADAPTIVE_BATCH_KEY,
    metrics=ADAPTIVE_METRICS,
    method=ADAPTIVE_METHOD,
)

print("Adaptive marking complete.")
# Show which adaptive outlier columns were created
adaptive_cols = [c for c in adata.obs.columns if "_adaptive" in c]
print(f"  Adaptive columns: {adaptive_cols}")

In [ ]:
print("\n---- Unified Cell Marking ----")

# Build the list of marking columns based on available doublet calls
marking_cols_to_plot = [
    "outlier_min_genes",
    "outlier_max_genes",
    "outlier_qc_metrics",
    "outlier_n_genes_by_counts_adaptive",
    "outlier_total_counts_adaptive",
    "outlier_mt",
    "outlier_pct_counts_mt_adaptive",
    "predicted_doublet",
]
# Add method-specific doublet columns if present
for dc in ["doubletdetection_predicted", "scrublet_predicted", "heuristic_predicted"]:
    if dc in adata.obs.columns:
        marking_cols_to_plot.append(dc)

marking_cfg = scl.qc.MarkingConfig(
    thresholds=scl.qc.QCThresholds(
        min_genes=final_min_genes,
        max_genes=final_max_genes,
        min_counts=final_min_counts,
        max_counts=final_max_counts,
        pc_mt=final_pc_mt,
        pc_top_genes={"pc_top_20_genes": 32.8},
        use_fixed_top_gene_threshold=True,
        nmads=MANUAL_NMADS,
    ),
    cols_to_plot=marking_cols_to_plot,
    save_dir=str(RESULTS_DIR / "qc_marking_plots"),
)

adata = scl.qc.mark_low_quality_cell(
    adata,
    sample_key=SAMPLE_KEY,
    config=marking_cfg,
)

# Summarize marking results
for col in marking_cols_to_plot:
    if col in adata.obs.columns:
        n = adata.obs[col].sum() if adata.obs[col].dtype == bool else "non-bool"
        if isinstance(n, (int, float)):
            print(f"  {col}: {n} cells flagged ({n/adata.n_obs*100:.1f}%)")

### 2.6 Filter Cells & Retention Audit

Applies the configured filter criteria and produces a detailed retention audit:
- Per-sample cell retention before/after
- Per-group retention (if group metadata available)
- Doublet-specific audit to confirm all doublets are removed


In [ ]:

# ==============================================================================
# QC FILTER CRITERIA
# ==============================================================================
# For tumor tissue, MT% is retained as review evidence by default instead of a
# standalone hard-removal criterion. Re-enable MT filtering only after inspecting
# per-sample distributions and treatment biology.
CRITERIA_TO_FILTER = [
    "outlier_min_genes",
    "outlier_qc_metrics",
    "outlier_n_genes_by_counts_adaptive",
    "outlier_total_counts_adaptive",
    # "outlier_mt",
    # "outlier_pct_counts_mt_adaptive",
    "predicted_doublet",
]
FILTER_COMBINATION = "any"  # "any" = remove if ANY criterion is True; "all" = ALL must be True


In [ ]:
print("\n---- Filtering Cells ----")

filter_cfg = scl.qc.FilterConfig(
    criteria_to_filter=CRITERIA_TO_FILTER,
    combination_logic=FILTER_COMBINATION,
)

adata_filtered = scl.qc.filter_cells(
    adata,
    config=filter_cfg,
    copy=True,
)

# ── Retention Audit ──
_ = scl.qc.audit_filtering(
    adata, adata_filtered, sample_key=SAMPLE_KEY, group_key=GROUP_KEY
)

# ── Doublet Audit ──
print("\n=== Doublet Audit ===")
audit_doublets(adata_filtered)

# Safety check
remaining_doublets = int(
    adata_filtered.obs.get("predicted_doublet", pd.Series(False, index=adata_filtered.obs_names)).sum()
)
if remaining_doublets > 0:
    print(f"\nWARNING: {remaining_doublets} predicted_doublet=True cells remain after filtering!")
    print("Review CRITERIA_TO_FILTER to ensure 'predicted_doublet' is included.")

### 2.7 Generate QC Reports

Two report formats are available:
1. **Standard text/CSV report** — `generate_qc_report()` — always generated
2. **Interactive HTML report** — `generate_qc_html_report()` — optional, requires plotly

The HTML report includes embedded stat cards, interactive Plotly charts, and
a complete record of QC decisions for reviewer traceability.


In [ ]:
print("\n---- Generating QC Reports ----")

# Standard report (always generated)
scl.qc.generate_qc_report(
    adata_filtered,
    save_dir=str(RESULTS_DIR / "qc_report"),
    sample_key=SAMPLE_KEY,
    include_before_after=True,
    adata_before=adata,
)

# Optional interactive HTML report
if GENERATE_HTML_REPORT:
    try:
        scl.qc.generate_qc_html_report(
            adata_filtered,
            output_path=str(RESULTS_DIR / "qc_html_report" / "qc_report.html"),
            adata_before=adata,
            title="scLucid Quality Control Report",
        )
        print("HTML QC report generated.")
    except ImportError:
        print("Skipping HTML report — plotly not available. Install with: pip install plotly")
    except Exception as e:
        print(f"HTML report generation failed: {e}")

print("\nQC reporting complete.")

### 2.8 QC Audit Summary

Final summary of QC results before moving to preprocessing.


In [ ]:
print("\n---- QC Audit Summary ----")
print(f"Before QC:  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
print(f"After QC:   {adata_filtered.n_obs:,} cells, {adata_filtered.n_vars:,} genes")
print(f"Removed:    {adata.n_obs - adata_filtered.n_obs:,} cells "
      f"({(1 - adata_filtered.n_obs/adata.n_obs)*100:.1f}%)")

print_sample_crosstab(adata_filtered, sample_key=SAMPLE_KEY, group_key=GROUP_KEY)
audit_doublets(adata_filtered)

In [ ]:
# Save QC-filtered data
adata_filtered.write(str(DATA_DIR / "Step1-sce_cleaned-CT26.h5ad"), compression="gzip")
print("\n---- QC Workflow Complete ----")
print(f"Saved: {DATA_DIR / 'Step1-sce_cleaned.h5ad'}")

---
## **Step 3: Preprocessing**

### Preprocessing strategy overview

1. **Gene biotype annotation** — classify genes as protein_coding, lncRNA, etc.
2. **Normalization** — log-normalize to 10K counts, store `.raw` for downstream DE
3. **Normalization diagnostics** — before/after plots to verify normalization effect
4. **Biotype filtering** — restrict to protein-coding (optional; `.raw` retains all genes)
5. **HVG selection** — identify informative genes with two complementary methods
6. **HVG stability** — bootstrap evaluation of HVG set robustness
7. **Regression & scaling** — regress technical covariates, scale to unit variance
8. **Scaling diagnostics** — verify regression and scaling effects
9. **Diagnostic embedding** — pre-integration UMAP to assess sample-driven structure
10. **Integration decision** — evaluate whether batch correction is needed
11. **Integration evaluation** — post-hoc metrics (iLISI, ASW, kBet)
12. **Final neighbors & UMAP** — optimized embedding on the final representation


In [ ]:

# ==============================================================================
# PREPROCESSING OPTIONS
# ==============================================================================
# Gene biotype
RUN_BIOTYPE_ANNOTATION = True
BIOTYPE_REFERENCE_PATH = None
BIOTYPE_METHOD = "custom"
KEEP_BIOTYPES = ["protein_coding"]
FILTER_BIOTYPE_AFTER_RAW = True

# Normalization
NORM_METHOD = "standard"
NORM_TARGET_SUM = 1e4
EXCLUDE_HIGHLY_EXPRESSED = True
USE_QUALITY_AWARE_NORM = False

# HVG selection — use two methods and take the union for multi-sample data.
HVG_N_TOP_GENES_SCANPY = 2500
HVG_N_TOP_GENES_CUSTOM = 1500
HVG_FLAVOR = "seurat"
HVG_UNION_MODE = "union"
HVG_MIN_N_SAMPLES = 2
RUN_HVG_STABILITY = True
HVG_PROTECTED_PRESETS = [
    "immune_receptor",
    "cytokine",
    "transcription_factor",
    "pathway",
    "tumor_heterogeneity",
]

# Regression
# Start with technical/library-size covariates. Cell-cycle regression is added
# only if diagnose_cell_cycle_regression() suggests a technical batch effect.
BASE_VARS_TO_REGRESS = ["total_counts", "pct_counts_mt"]
VARS_TO_REGRESS = list(BASE_VARS_TO_REGRESS)
REGRESS_IN_SCALE = False
SCALE_MAX_VALUE = 10

# PCA
N_PCS_MAX = 50

# Neighbors optimization
N_NEIGHBORS_LIST = [15, 20, 25, 30]
N_PCS_LIST = [20, 25, 30, 35, 40, 45, 50]
N_JOBS = -1

# Integration
# MODEL_KEY stores biological model identity (e.g. CT26/SCC7) and should not be used
# as a technical integration key. In model-split analyses, sampleID is the only
# candidate integration key, and auto mode will still skip unsafe confounded cases.
RUN_INTEGRATION = "auto"
INTEGRATION_METHOD = "harmony"
INTEGRATION_BATCH_KEY = SAMPLE_KEY
EVALUATE_INTEGRATION = True

# Embedding optimization on final representation
RUN_EMBEDDING_OPTIMIZATION = True

# ==============================================================================
# CHECKPOINT & RESUME
# ==============================================================================
USE_CHECKPOINTS = False
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"

# ==============================================================================
# DIAGNOSTIC PLOTS
# ==============================================================================
SHOW_NORMALIZATION_DIAGNOSTICS = True
SHOW_SCALING_DIAGNOSTICS = True
SHOW_INTEGRATION_DIAGNOSTICS = True

In [ ]:

# Load QC-filtered data
adata = sc.read_h5ad(str(DATA_DIR / "Step1-sce_cleaned-CT26.h5ad"))
print(f"Dataset loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes")

# Ensure counts layer exists without overwriting a valid existing raw-count layer.
if "counts" in adata.layers:
    raw_like, count_diagnostics = is_raw_like_counts_matrix(adata.layers["counts"])
    print(f"Existing counts layer detected. raw_like={raw_like}; diagnostics={count_diagnostics}")
else:
    raw_like, count_diagnostics = is_raw_like_counts_matrix(adata.X)
    print(f"No counts layer found. X raw_like={raw_like}; diagnostics={count_diagnostics}")
    if not raw_like:
        raise ValueError(
            "Cannot safely create adata.layers['counts'] from adata.X because X does not look like raw counts. "
            "Reload the raw/QC object that preserves counts before preprocessing."
        )
    adata.layers["counts"] = adata.X.copy()
    print("Created adata.layers['counts'] from raw-like adata.X.")

print(f"\n{'='*60}")
print("Preprocessing")
print(f"{'='*60}")


### 3.1 Gene Biotype Annotation

Annotate genes by biotype (protein_coding, lncRNA, pseudogene, etc.).
Two modes:
- **reference**: use scLucid's bundled reference (Ensembl)
- **custom**: provide a gene_info file with `gene_name` and `biotype` columns

Annotation is done **before** filtering so that `.raw` can retain all genes.


In [ ]:
if RUN_BIOTYPE_ANNOTATION:
    if BIOTYPE_METHOD == "custom" and BIOTYPE_REFERENCE_PATH is not None:
        gene_info = pd.read_csv(BIOTYPE_REFERENCE_PATH, sep="\t")
        biotype_df = (
            gene_info[["external_gene_name", "gene_biotype"]]
            .dropna()
            .drop_duplicates()
            .rename(columns={
                "external_gene_name": "gene_name",
                "gene_biotype": "biotype",
            })
        )
        adata = scl.pp.annotate_gene_biotypes(
            adata,
            biotype_df=biotype_df,
            method="custom",
            fuzzy_match=True,
            overwrite=True,
        )
    else:
        # Use scLucid bundled reference
        adata = scl.pp.apply_gene_biotype_strategy(
            adata,
            species=SPECIES,
            method="reference",
            do_filter=False,  # Don't filter yet — do it after .raw is set
            fuzzy_match=True,
            overwrite=True,
            copy=False,
        )

    print("Biotype annotation complete.")
    print(adata.var["biotype_category"].value_counts())
else:
    print("Gene biotype annotation skipped (RUN_BIOTYPE_ANNOTATION = False).")

### 3.2 Normalization & `.raw` Setup

**Critical ordering**: Normalize first, then set `.raw`, then optionally filter biotypes.

- `normalize_data()` — library-size normalization (log1p after size-factor scaling)
- `.raw` — stores the **full-gene normalized expression** for downstream differential
  expression testing and marker inspection
- Quality-aware normalization: when `USE_QUALITY_AWARE_NORM = True`, normalization
  adapts to per-cell QC scores, down-weighting low-quality cells

Why `.raw` before biotype filtering:
- Differential expression and marker gene inspection need access to all genes,
  not just the protein-coding subset used for clustering.
- Setting `.raw` before filtering preserves this full-gene reference.


In [ ]:

print("\n##====== 1. Normalizing Data & Setting .raw ======##")

if USE_QUALITY_AWARE_NORM:
    print("Using quality-aware normalization...")
    requested_quality_metrics = ["pct_counts_mt", "algorithm_doublet_score", "doublet_score"]
    available_quality_metrics = [m for m in requested_quality_metrics if m in adata.obs.columns]
    print(f"Quality metrics used: {available_quality_metrics}")
    adata = scl.pp.quality_aware_normalize(
        adata,
        quality_metrics=available_quality_metrics,
        input_layer="counts",
        output_layer="normalized",
        target_sum=NORM_TARGET_SUM,
    )
else:
    adata = scl.pp.normalize_data(
        adata,
        config=scl.pp.NormalizationConfig(
            method=NORM_METHOD,
            target_sum=NORM_TARGET_SUM,
            exclude_highly_expressed=EXCLUDE_HIGHLY_EXPRESSED,
            input_layer="counts",
            output_layer="normalized",
            save_dir=str(RESULTS_DIR / "pp_normalization"),
        ),
    )

# Store full-gene normalized expression in .raw BEFORE any gene filtering.
# This is essential for downstream DE and marker inspection.
print("Storing normalized full-gene expression in adata.raw before biotype filtering...")
adata.raw = adata.copy()
print(f"  adata.raw shape: {adata.raw.shape}")


### 3.3 Normalization Diagnostics

Visualize the effect of normalization: before vs. after distributions,
mean-variance relationships, and library-size normalization confirmation.


In [ ]:
if SHOW_NORMALIZATION_DIAGNOSTICS:
    try:
        scl.pp.plot_normalization_effect(
            adata,
            original_layer="counts",
            normalized_layer="normalized" if "normalized" in adata.layers else None,
            save_dir=str(RESULTS_DIR / "pp_normalization"),
        )
        print("Normalization diagnostics generated.")
    except Exception as e:
        print(f"Normalization diagnostics skipped: {e}")
else:
    print("Normalization diagnostics skipped (SHOW_NORMALIZATION_DIAGNOSTICS = False).")

### 3.4 Gene Biotype Filtering (Optional)

Filter genes to retain only the specified biotypes (default: protein_coding).
This is applied AFTER `.raw` is set, so `.raw` retains all genes for downstream DE.

If `FILTER_BIOTYPE_AFTER_RAW = False`, this step is skipped.


In [ ]:
if FILTER_BIOTYPE_AFTER_RAW and RUN_BIOTYPE_ANNOTATION:
    print("Filtering genes by biotype...")
    adata = scl.pp.filter_genes_by_biotype(
        adata,
        keep_biotypes=KEEP_BIOTYPES,
        copy=True,
    )
    print(f"After biotype filtering: {adata.n_vars:,} genes remaining.")
    print(adata.var["biotype_category"].value_counts())
else:
    print("Biotype filtering skipped.")

### 3.5 Highly Variable Gene (HVG) Selection

Two complementary methods, combined via union for multi-sample robustness:

1. **scanpy (seurat flavor)** — standard HVG based on mean-variance relationship.
   Good default, but can miss sample-specific signals in multi-sample data.
2. **custom (sample-aware)** — identifies genes that are highly variable within
   individual samples and highly expressed across samples. Better for
   heterogeneous multi-sample designs.

The `suggest_hvg_choice()` function guides the combination strategy.


In [ ]:

print("\n##====== 2. Highly Variable Gene (HVG) Selection ======##")

HVG_SCANPY_KEY = f"highly_variable_scanpy_{HVG_FLAVOR}"
HVG_CUSTOM_KEY = "highly_variable_custom"
HVG_KEYS = [HVG_SCANPY_KEY, HVG_CUSTOM_KEY]

# Method 1: Standard scanpy HVG with biology-protected presets for tumor/immune analysis.
adata = scl.pp.find_hvgs(
    adata,
    input_layer="normalized",
    config=scl.pp.HVGConfig(
        method="scanpy",
        flavor=HVG_FLAVOR,
        n_top_genes=HVG_N_TOP_GENES_SCANPY,
        span=0.5,
        protected_gene_presets=HVG_PROTECTED_PRESETS,
        protection_max_extra_genes=300,
    ),
)

# Method 2: Sample-aware custom HVG with the same protection policy.
adata = scl.pp.find_hvgs(
    adata,
    input_layer="normalized",
    config=scl.pp.HVGConfig(
        method="custom",
        sample_key=SAMPLE_KEY,
        n_top_genes=HVG_N_TOP_GENES_CUSTOM,
        min_n_samples=HVG_MIN_N_SAMPLES,
        n_highly_expressed_genes=50,
        n_specific_genes=10,
        protected_gene_presets=HVG_PROTECTED_PRESETS,
        protection_max_extra_genes=300,
    ),
)

missing_hvg_keys = [key for key in HVG_KEYS if key not in adata.var.columns]
if missing_hvg_keys:
    raise KeyError(f"Expected HVG mask(s) missing from adata.var: {missing_hvg_keys}")

# Compare and guide the combination. suggest_hvg_choice returns structured guidance;
# printing remains a notebook concern.
print("\n--- HVG Set Comparison ---")
hvg_choice_suggestion = scl.pp.suggest_hvg_choice(
    adata,
    hvg_keys=HVG_KEYS,
    mode=HVG_UNION_MODE,
)
print("\n".join(hvg_choice_suggestion.get("messages", [])))


#### HVG Stability Evaluation

Bootstrapping-based stability assessment: resamples cells and recomputes HVG
to measure how robust each gene's "highly variable" designation is.

This is especially useful for:
- Heterogeneous samples where HVG sets may be unstable
- Publications where method justification is required
- Deciding between union vs. intersection for combining HVG sets


In [ ]:
if RUN_HVG_STABILITY:
    print("\n--- HVG Stability Evaluation ---")
    try:
        _ = scl.pp.evaluate_hvg_stability(
            adata,
            hvg_key=HVG_SCANPY_KEY,
            n_bootstrap=20,
            flavor=HVG_FLAVOR,
            n_top_genes=HVG_N_TOP_GENES_SCANPY,
        )
        print("HVG stability evaluation complete.")
        hvg_stability = adata.uns.get("sclucid", {}).get("preprocess", {}).get("hvg_stability")
        print("  HVG stability metrics stored in adata.uns['sclucid']['preprocess']['hvg_stability']")
        if hvg_stability is not None:
            print(f"  Stability summary keys: {list(hvg_stability.keys())}")
    except Exception as e:
        print(f"HVG stability evaluation skipped: {e}")
else:
    print("HVG stability evaluation skipped (RUN_HVG_STABILITY = False).")


In [ ]:
# Select final HVG set with a compact audit trail.
adata, hvg_selection_audit = scl.pp.select_and_audit_hvgs(
    adata,
    hvg_keys=HVG_KEYS,
    mode=HVG_UNION_MODE,
    subset=True,
    plot_venn=True,
    keep_raw=False,  # .raw was already set manually after normalization
    evaluate_stability=False,  # stability was evaluated explicitly above when enabled
    save_dir=str(RESULTS_DIR / "pp_hvg"),
)

print()
print(f"Final HVG set: {adata.n_vars:,} genes")
print("HVG audit:", hvg_selection_audit)


### 3.6 Regression, Scaling & PCA

Sequential steps on the HVG subset:

1. **Regression** — regress out technical covariates (total_counts, pct_mt, cell cycle scores)
   from the normalized expression. This removes technical variation without distorting
   biological signal.
2. **Scaling** — z-score scaling to unit variance, capped at `max_value` to prevent
   extreme values from dominating the PCA.
3. **PCA** — dimensionality reduction to `min(50, n_cells-1, n_genes-1)` components.

Note on cell cycle regression:
If the biological process of interest may involve proliferation, consider running
a sensitivity analysis without cell cycle regression.


In [ ]:

print("\n##====== 3. Regression, Scaling & PCA ======##")

cc_regression_diagnostic = scl.pp.diagnose_cell_cycle_regression(
    adata,
    condition_key=GROUP_KEY if GROUP_KEY in adata.obs.columns else None,
    batch_key=INTEGRATION_BATCH_KEY if INTEGRATION_BATCH_KEY in adata.obs.columns else None,
    tumor=("tumor" in str(TISSUE_TYPE).lower()),
)
print("Cell-cycle regression diagnostic:")
print(f"  status: {cc_regression_diagnostic.get('status')}")
print(f"  recommendation: {cc_regression_diagnostic.get('recommendation')}")
for warning in cc_regression_diagnostic.get("warnings", []):
    print(f"  ! {warning}")

VARS_TO_REGRESS = [v for v in BASE_VARS_TO_REGRESS if v in adata.obs.columns]
if cc_regression_diagnostic.get("status") == "technical_regression_candidate":
    VARS_TO_REGRESS.extend([v for v in ["S_score", "G2M_score"] if v in adata.obs.columns])

print(f"Regressing out covariates: {VARS_TO_REGRESS}")
if VARS_TO_REGRESS:
    regress_cfg = scl.pp.ScalingConfig(
        vars_to_regress=VARS_TO_REGRESS,
        regress_in_scale=REGRESS_IN_SCALE,
        plot=False,
        report=False,
    )
    adata = scl.pp.regress_out(
        adata,
        config=regress_cfg,
        input_layer="normalized",
        output_layer="regressed",
    )
else:
    adata.layers["regressed"] = adata.layers["normalized"].copy()

# Scaling
adata.X = adata.layers["regressed"].copy()
scale_cfg = scl.pp.ScalingConfig(
    regress_in_scale=False,
    max_value=SCALE_MAX_VALUE,
    plot=False,
    report=False,
)
adata = scl.pp.scale_data(adata, config=scale_cfg, output_layer="scaled")
adata.X = adata.layers["scaled"].copy()

# PCA
n_pcs = min(N_PCS_MAX, adata.n_obs - 1, adata.n_vars - 1)
sc.tl.pca(adata, n_comps=n_pcs)
print(f"PCA completed: {n_pcs} components.")

### 3.7 Scaling Diagnostics

Visualize the effect of regression and scaling:
- Before/after distributions of expression values
- PCA elbow plot
- Variance explained by technical vs. biological factors


In [ ]:
if SHOW_SCALING_DIAGNOSTICS:
    print("\n--- Scaling Diagnostics ---")
    try:
        scl.pp.plot_scaling_effect(
            adata,
            original_data=adata.layers["regressed"],
            scaled_layer="scaled",
            save_dir=str(RESULTS_DIR / "pp_scaling"),
        )
        print("Scaling diagnostics generated.")
    except Exception as e:
        print(f"Scaling diagnostics skipped: {e}")
else:
    print("Scaling diagnostics skipped (SHOW_SCALING_DIAGNOSTICS = False).")

### 3.8 Diagnostic Embedding (Pre-Integration)

Build a diagnostic UMAP using the unintegrated PCA to visualize sample-driven
structure **before** any batch correction. This serves as a baseline:
- If samples already mix well → integration may not be needed
- If samples form distinct clusters → integration may help visualize shared biology
- If samples separate by group → the batch and biology keys may be confounded


In [ ]:
print()
print("##====== 4. Diagnostic Embedding (Pre-Integration) ======##")

optimization_results = scl.pp.optimize_neighbors_pcs(
    adata,
    config=scl.pp.NeighborsConfig(
        use_rep="X_pca",
        n_neighbors_list=N_NEIGHBORS_LIST,
        n_pcs_list=N_PCS_LIST,
        n_jobs=N_JOBS,
    ),
)

if len(optimization_results) > 0:
    best_params = optimization_results.loc[optimization_results["silhouette_score"].idxmax()]
    diagnostic_n_neighbors = int(best_params["n_neighbors"])
    diagnostic_n_pcs = int(best_params["n_pcs"])
    print(
        f"Diagnostic PCA embedding: n_neighbors={diagnostic_n_neighbors}, "
        f"n_pcs={diagnostic_n_pcs}, silhouette={best_params['silhouette_score']:.4f}"
    )
    display(optimization_results)
else:
    diagnostic_n_neighbors = 15
    diagnostic_n_pcs = min(50, n_pcs)
    print(f"No valid optimization result; using defaults n_neighbors={diagnostic_n_neighbors}, n_pcs={diagnostic_n_pcs}.")

sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=diagnostic_n_neighbors, n_pcs=diagnostic_n_pcs)
sc.tl.umap(adata)
adata.obsm["X_umap_pca"] = adata.obsm["X_umap"].copy()
adata.uns.setdefault("sclucid", {}).setdefault("preprocess", {})["embedding_workflow"] = {
    "use_rep": "X_pca",
    "umap_key": "X_umap_pca",
    "optimized": bool(len(optimization_results) > 0),
    "n_neighbors": diagnostic_n_neighbors,
    "n_pcs": diagnostic_n_pcs,
    "min_dist": None,
}


In [ ]:
# MODEL_KEY is biological model metadata. Do not use it as an integration key.

In [ ]:
# Plot diagnostic pre-integration UMAP
plot_keys = [SAMPLE_KEY]
if GROUP_KEY and GROUP_KEY in adata.obs.columns:
    plot_keys.append(GROUP_KEY)
if MODEL_KEY and MODEL_KEY in adata.obs.columns:
    plot_keys.append(MODEL_KEY)

palette = scl.pl.build_obs_palette(
    adata,
    plot_keys,
    color_maps={
        "samples": SAMPLE_COLORS,
        "groups": GROUP_COLORS,
        "models": MODEL_COLORS,
    },
)

sc.pl.embedding(
    adata,
    basis="X_umap_pca",
    color=plot_keys,
    title=[f"Pre-Integration: {k}" for k in plot_keys],
    palette=palette,
    ncols=3,
    frameon="small",
    show=True,
    wspace=0.45,
)
print("Diagnostic pre-integration UMAP plotted (X_umap_pca).")

### 3.9 Integration Decision & Batch Correction

**Auto-detection logic**:
- `MODEL_KEY` stores biological model identity (e.g. CT26/SCC7) and is not used for integration.
- If `RUN_INTEGRATION = "auto"` → evaluate `INTEGRATION_BATCH_KEY` against protected biology columns.
- If the candidate integration key is confounded with biology → integration is **skipped** to avoid
  removing the biological signal of interest.
- If the candidate key is independent of protected biology → integration proceeds.

When integration is run, the result is stored in a named key (e.g. `X_harmony_sampleID`)
to distinguish it from unintegrated `X_pca`. The **final analysis graph** should use
the representation that preserves biological signal.

In [ ]:
print("\n##====== 5. Integration Decision ======##")

integration_biology_columns = [c for c in BIOLOGY_COLUMNS if c and c in adata.obs.columns]
integration_condition_key = GROUP_KEY if GROUP_KEY in adata.obs.columns else None
integration_is_tumor = "tumor" in str(TISSUE_TYPE).lower()

run_integration, integration_warnings, integration_risk = scl.pp.decide_integration(
    adata,
    batch_key=INTEGRATION_BATCH_KEY,
    run_integration=RUN_INTEGRATION,
    biology_columns=integration_biology_columns,
    condition_key=integration_condition_key,
    tumor=integration_is_tumor,
    before_rep="X_pca",
)

print(f"Integration candidate key: '{INTEGRATION_BATCH_KEY}'")
print(f"Model metadata key: '{MODEL_KEY}' (not used for integration)")
print(f"Protected biology columns: {BIOLOGY_COLUMNS}")
print(f"Integration needed: {run_integration}")
if integration_risk is not None:
    print(f"Integration risk: {integration_risk.get('risk_level')} ({integration_risk.get('risk_score')})")
    print(f"Recommendation: {integration_risk.get('recommendation')}")

if integration_warnings:
    print("\nIntegration warnings:")
    for w in integration_warnings:
        print(f"  ! {w}")

if run_integration:
    output_key = f"X_{INTEGRATION_METHOD}_{INTEGRATION_BATCH_KEY}"
    print(f"\nRunning {INTEGRATION_METHOD} integration on '{INTEGRATION_BATCH_KEY}'...")
    print(f"Output key: {output_key}")

    adata = scl.pp.batch_correction(
        adata,
        config=scl.pp.IntegrationConfig(
            method=INTEGRATION_METHOD,
            batch_key=INTEGRATION_BATCH_KEY,
            use_rep="X_pca",
            output_key=output_key,
            auto_decide=(RUN_INTEGRATION == "auto"),
            condition_key=integration_condition_key,
            biology_columns=integration_biology_columns,
            tumor=integration_is_tumor,
            evaluate=False,
        ),
        plot=False,
        force=True,
    )
    print(f"Integration complete. Embedding stored in adata.obsm['{output_key}'].")
else:
    output_key = "X_pca"
    print("\nSkipping integration — using unintegrated X_pca as final representation.")
    if integration_warnings:
        print("Reason(s): " + "; ".join(integration_warnings))


### 3.10 Integration Quality Evaluation

If integration was run, evaluate its quality using established metrics:
- **iLISI** (integration Local Inverse Simpson's Index) — measures batch mixing
- **cLISI** (cell-type LISI) — measures cell-type separation
- **ASW** (Average Silhouette Width) — batch-wise vs. cluster-wise
- **kBet** — k-nearest neighbor batch effect test

Good integration: high iLISI, high ASW_cluster, low ASW_batch.


In [ ]:
INTEGRATION_BATCH_KEY

In [ ]:

if EVALUATE_INTEGRATION and run_integration and output_key != "X_pca":
    print("\n--- Integration Quality Evaluation ---")
    try:
        integration_metrics = scl.pp.evaluate_integration(
            adata,
            batch_key=INTEGRATION_BATCH_KEY,
            use_rep=output_key,
        )
        print("Integration quality metrics:")
        for metric_name, metric_value in integration_metrics.items():
            if metric_name == "interpretation":
                continue
            if isinstance(metric_value, (int, float)):
                print(f"  {metric_name}: {metric_value:.4f}")
            else:
                print(f"  {metric_name}: {metric_value}")

        interpretation = integration_metrics.get("interpretation", {})
        if interpretation:
            print(f"Integration interpretation: {interpretation.get('status')}")
            print(f"Recommendation: {interpretation.get('recommendation')}")
            for warning in interpretation.get("warnings", []):
                print(f"  ! {warning}")

        post_integration_risk = scl.pp.diagnose_integration_risk(
            adata,
            batch_key=INTEGRATION_BATCH_KEY,
            condition_key=integration_condition_key,
            biology_columns=integration_biology_columns,
            tumor=integration_is_tumor,
            before_rep="X_pca",
            after_rep=output_key,
            label_key=None,
            key_added="post_integration_risk",
        )
        print(f"Post-integration risk: {post_integration_risk.get('risk_level')}")
    except Exception as e:
        print(f"Integration evaluation skipped: {e}")
elif not run_integration:
    print("Skipping integration evaluation — no integration was run.")
else:
    print("Skipping integration evaluation (EVALUATE_INTEGRATION = False).")


### 3.11 Final Neighbors & UMAP Optimization

Optimize neighbors and UMAP on the final representation (integrated or unintegrated).
If embedding optimization is enabled, scans over the parameter grid and selects
the combination with the highest silhouette score.


In [ ]:
adata

In [ ]:
print()
print("##====== 6. Final Graph & UMAP ======##")
print(f"Final representation: {output_key}")

final_umap_key = f"X_umap_{output_key.replace('X_', '').lower()}"

if RUN_EMBEDDING_OPTIMIZATION:
    print("Running embedding optimization...")
    optimization_results = scl.pp.optimize_neighbors_pcs(
        adata,
        config=scl.pp.NeighborsConfig(
            use_rep=output_key,
            n_neighbors_list=N_NEIGHBORS_LIST,
            n_pcs_list=N_PCS_LIST,
            n_jobs=N_JOBS,
        ),
    )

    if len(optimization_results) > 0:
        best_params = optimization_results.loc[optimization_results["silhouette_score"].idxmax()]
        best_n_neighbors = int(best_params["n_neighbors"])
        best_n_pcs = int(best_params["n_pcs"])
        print(
            f"Optimized: n_neighbors={best_n_neighbors}, n_pcs={best_n_pcs} "
            f"(silhouette={best_params['silhouette_score']:.4f})"
        )
        display(optimization_results)
    else:
        print("No valid optimization result; using defaults.")
        best_n_neighbors = 15
        best_n_pcs = min(50, n_pcs)
else:
    optimization_results = pd.DataFrame()
    best_n_neighbors = 15
    best_n_pcs = min(50, n_pcs)
    print(f"Using default parameters: n_neighbors={best_n_neighbors}, n_pcs={best_n_pcs}")

sc.pp.neighbors(adata, use_rep=output_key, n_neighbors=best_n_neighbors, n_pcs=best_n_pcs)
sc.tl.umap(adata, min_dist=0.3)
adata.obsm[final_umap_key] = adata.obsm["X_umap"].copy()
adata.uns.setdefault("sclucid", {}).setdefault("preprocess", {})["embedding_workflow"] = {
    "use_rep": output_key,
    "umap_key": final_umap_key,
    "optimized": bool(RUN_EMBEDDING_OPTIMIZATION and len(optimization_results) > 0),
    "n_neighbors": best_n_neighbors,
    "n_pcs": best_n_pcs,
    "min_dist": 0.3,
}
print(f"Final UMAP saved as adata.obsm['{final_umap_key}']")


In [ ]:
print("\n##====== 7. Plotting Final Embedding ======##")

plot_keys = [SAMPLE_KEY]
if GROUP_KEY and GROUP_KEY in adata.obs.columns:
    plot_keys.append(GROUP_KEY)
if MODEL_KEY and MODEL_KEY in adata.obs.columns:
    plot_keys.append(MODEL_KEY)

palette = scl.pl.build_obs_palette(
    adata,
    plot_keys,
    color_maps={
        "samples": SAMPLE_COLORS,
        "groups": GROUP_COLORS,
        "models": MODEL_COLORS,
    },
)

sc.pl.embedding(
    adata,
    basis=final_umap_key,
    color=plot_keys,
    title=[f"Final: {k}" for k in plot_keys],
    wspace=0.4,
    palette=palette,
    show=True,
)

---
## **Step 4: Contract Validation & Final Audit**

scLucid defines **stage contracts** — formal specifications of what each analysis
stage should produce in the AnnData object. This cell validates:

1. **Layer consistency** — required layers are present (`counts`, `normalized`, `regressed`, `scaled`)
2. **`.raw` integrity** — `.raw` should retain all genes from normalization, before HVG/biotype filtering
3. **Embedding keys** — expected embeddings are present
4. **Stage contracts** — formal validation of QC and preprocessing stage outputs

If validation fails, review the workflow steps. Non-critical warnings about
`review_summary` entries may appear when using modular (non-workflow) API calls;
these are provenance notes, not biological errors.


In [ ]:

print("\n##====== 8. Final Object Audit ======##")
print(f"Final shape: {adata.shape}")
print(f"Layers: {list(adata.layers.keys())}")
print(f"Raw shape: {adata.raw.shape if adata.raw is not None else 'MISSING — DE and marker inspection may be compromised'}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"sclucid uns keys: {list(adata.uns.get('sclucid', {}).keys())}")

# Layer integrity
required_layers = {"counts", "normalized", "regressed", "scaled"}
missing_layers = required_layers - set(adata.layers.keys())
if missing_layers:
    print(f"WARNING: Missing expected layers: {missing_layers}")
else:
    print("All required layers present.")

# .raw integrity
if adata.raw is None:
    print("WARNING: adata.raw is None — DE testing will be limited to HVG subset only.")
elif adata.raw.shape[1] <= adata.shape[1]:
    print("WARNING: adata.raw has <= genes than filtered adata — raw should retain all normalized genes before HVG/biotype filtering.")

# Embedding sanity check
if "X_harmony" in adata.obsm and output_key != "X_harmony":
    print(f"NOTE: Generic 'X_harmony' found but final representation is '{output_key}'. Plotting should use '{final_umap_key}'.")

# Finalize manual review summaries for the audit-grade contract.
try:
    qc_review_summary = scl.ut.finalize_manual_review_summary(
        adata,
        module="qc",
        workflow_name="AK112_step1_manual_qc_refined",
        steps=[
            "load_10x_data",
            "calculate_qc_metric",
            "diagnose_ambient_rna",
            "diagnose_empty_droplets",
            "recommend_intelligent_qc",
            "predict_doublets",
            "suggest_qc_thresholds",
            "mark_low_quality_cells",
            "filter_cells",
            "generate_qc_report",
        ],
        config={
            "sample_key": SAMPLE_KEY,
            "group_key": GROUP_KEY,
            "model_key": MODEL_KEY,
            "integration_candidate_key": INTEGRATION_BATCH_KEY,
            "doublet_method": DOUBLET_METHOD,
            "filter_criteria": CRITERIA_TO_FILTER,
            "mt_filtering_policy": "review_evidence_not_default_hard_filter",
        },
        summary={
            "ambient_summary": adata.uns.get("sclucid", {}).get("qc", {}).get("ambient_summary", {}),
            "empty_droplet_summary": adata.uns.get("sclucid", {}).get("qc", {}).get("empty_droplet_summary", {}),
            "final_thresholds": {
                "min_genes": final_min_genes,
                "min_counts": final_min_counts,
                "max_genes": final_max_genes,
                "max_counts": final_max_counts,
                "pc_mt": final_pc_mt,
            },
        },
        save_dir=str(RESULTS_DIR / "qc_review_summary"),
        warnings=[
            "Manual modular QC path: review plots and per-sample retention before treating filters as final.",
            "Tumor tissue MT% retained as review evidence by default.",
        ],
        title="AK112 Step1 QC Review Summary",
    )
    preprocess_review_summary = scl.ut.finalize_manual_review_summary(
        adata,
        module="preprocess",
        workflow_name="AK112_step1_manual_preprocess_refined",
        steps=[
            "raw_count_semantics_guard",
            "gene_biotype_annotation",
            "normalize_data",
            "set_raw_before_gene_filtering",
            "protected_hvg_selection",
            "cell_cycle_regression_diagnostic",
            "scaling",
            "pca",
            "integration_risk_diagnostic",
            "neighbors_umap",
        ],
        config={
            "normalization_method": NORM_METHOD,
            "hvg_union_mode": HVG_UNION_MODE,
            "hvg_protected_presets": HVG_PROTECTED_PRESETS,
            "vars_to_regress": VARS_TO_REGRESS,
            "integration_candidate_key": INTEGRATION_BATCH_KEY,
            "model_key": MODEL_KEY,
            "final_representation": output_key,
        },
        summary={
            "cell_cycle_regression_diagnostic": adata.uns.get("sclucid", {}).get("preprocess", {}).get("cell_cycle_regression_diagnostic", {}),
            "integration": adata.uns.get("sclucid", {}).get("preprocess", {}).get("integration", {}),
            "final_umap_key": final_umap_key,
        },
        save_dir=str(RESULTS_DIR / "preprocess_review_summary"),
        warnings=[
            "Cell-cycle regression is conditional and conservative for tumor tissue.",
            "Batch integration is skipped unless a true multi-level technical batch passes risk diagnostics.",
        ],
        title="AK112 Step1 Preprocess Review Summary",
    )
    print("Manual review summaries finalized.")
except Exception as exc:
    print(f"Manual review summary finalization skipped due to error: {exc}")

# Stage contract validation
try:
    from scLucid.utils import validate_all_stage_contracts
    contract_results = validate_all_stage_contracts(adata, when="output")
    print("\n=== Stage Contract Validation ===")
    for stage, result in contract_results.items():
        status = "PASS" if result.valid else "FAIL"
        print(f"  {stage}: {status}")
        if result.errors:
            for err in result.errors:
                print(f"    - {err}")
except Exception as exc:
    print(f"Contract validation skipped due to error: {exc}")

# Final check
print(f"\n{'='*60}")
print(f"Preprocessing complete.")
print(f"  Cells:  {adata.n_obs:,}")
print(f"  Genes:  {adata.n_vars:,}")
print(f"  Final embedding: {final_umap_key}")
print(f"{'='*60}")

In [ ]:
# Save final preprocessed AnnData through scLucid's safe writer.
# Full mode keeps X + counts/normalized/regressed/scaled + raw for auditability.
# Lightweight mode drops dense working layers (regressed/scaled) and writes X from normalized.
SAVE_LIGHTWEIGHT_PREPROCESSED = False
output_path = DATA_DIR / "Step2-sce_preprocessed-CT26.h5ad"
if SAVE_LIGHTWEIGHT_PREPROCESSED:
    output_path = output_path.with_name(output_path.stem + "_light.h5ad")

saved_path = scl.ut.write_h5ad_safe(
    adata,
    output_path,
    compression="gzip",
    sanitize_uns=True,
    atomic=True,
    lightweight=SAVE_LIGHTWEIGHT_PREPROCESSED,
    drop_layers=("regressed", "scaled"),
    x_layer="normalized" if SAVE_LIGHTWEIGHT_PREPROCESSED else None,
)
print(f"\nSaved: {saved_path}")
print("\n---- Preprocessing Complete ----")
